In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

cwd = Path.cwd().resolve()

def find_release_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        data_root = candidate / 'Step0_Data' / '1_Data'
        if (data_root / 'All_Waste_182countries_1990_2024.csv').is_file():
            return candidate
    raise FileNotFoundError('Could not locate standalone release root from the current working directory')

release_root = find_release_root(cwd)
shared_data_root = release_root / 'Step0_Data' / '1_Data'
step_root = release_root / 'Step0_Data'
input_path = shared_data_root / 'All_Waste_182countries_1990_2024.csv'
output_path = step_root / '3_Results' / '0_cleaned_baseline.csv'
summary_path = step_root / '3_Results' / '0_cleaning_summary.csv'
waste_cols = ['AW_t', 'CDW_t', 'IW_t', 'MSW_t']
output_path

In [ ]:
df = pd.read_csv(input_path)
df = df.sort_values(['ISO3', 'Year']).reset_index(drop=True)
zero_counts = {col: int((pd.to_numeric(df[col], errors='coerce') == 0).sum()) for col in waste_cols}
zero_counts

In [ ]:
for col in waste_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').replace(0, np.nan)
null_counts = df[waste_cols].isna().sum().to_dict()
null_counts

In [ ]:
output_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output_path, index=False)
pd.DataFrame({
    'column': waste_cols,
    'zero_replaced': [zero_counts[col] for col in waste_cols],
    'null_after_clean': [int(df[col].isna().sum()) for col in waste_cols],
}).to_csv(summary_path, index=False)
summary_path